# DATA2001 Full Workflow


## 0. Team Information


| name | unikey | SA4 Zone |
|------|--------|----------|
| xuejian fang | xfan0282 | Sydney - City and Inner South |
|
|
|

## 1. Environment and Configuration

This section imports the dependencies and project interfaces used by the rest of the workflow, loads the YAML configuration, and creates the database engine for later steps.

> Prerequisites: follow the README first to prepare the environment. Make sure the local PostgreSQL/PostGIS container is running, run `uv sync` from the project root to create the virtual environment, and select that virtual environment as the Python kernel for this notebook.


In [2]:
from pathlib import Path

import pandas as pd
import plotly.express as px

from data2001.config import load_settings
from data2001.db.engine import create_engine_from_settings
from data2001.pipeline import execute_workflow_steps
from data2001.task1_cleaning.workflow import run_task1_cleaning
from data2001.task1_statistics.workflow import run_all_task1_statistics
from data2001.task4.queries import (
    expected_report_figure_paths,
    load_api_extraction_summary,
    load_correlation_results,
    load_correlation_summary,
    load_index_summary,
    load_poi_group_counts,
    load_poi_points,
    load_sa2_scores,
    load_schema_summary,
    load_score_income,
    load_score_input_summary,
    load_spatial_join_summary,
    load_table_counts,
)
from data2001.task4.charts import (
    build_bottom_sa2_bar,
    build_poi_group_distribution,
    build_score_histogram,
    build_score_income_scatter,
    build_top_sa2_bar,
)
from data2001.task4.maps import build_poi_point_scatter_map, build_score_choropleth_map
from data2001.task4.tables import build_sa4_summary_table, build_top_bottom_table


settings = load_settings("configs/local.yaml")
engine = create_engine_from_settings(settings.database)

settings


Settings(database=DatabaseSettings(driver='postgresql+psycopg', host='localhost', port=5432, database='data2001', user='data2001', password='data2001', schema_name='data2001'), api=APISettings(layers={'poi': LayerSettings(url='https://maps.six.nsw.gov.au/arcgis/rest/services/public/NSW_POI/MapServer/0/query', geometry_type='esriGeometryPoint', expected_srid=4283, expected_fields=['objectid', 'topoid', 'poigroup', 'poitype', 'poiname', 'poilabel', 'poilabeltype', 'poialtlabel', 'poisourcefeatureoid', 'accesscontrol', 'startdate', 'enddate', 'lastupdate', 'msoid', 'centroidid', 'shapeuuid', 'changetype', 'processstate', 'urbanity'], out_fields=['objectid', 'topoid', 'poigroup', 'poitype', 'poiname', 'poilabel', 'poilabeltype', 'poialtlabel', 'poisourcefeatureoid', 'accesscontrol', 'startdate', 'enddate', 'lastupdate', 'msoid', 'centroidid', 'shapeuuid', 'changetype', 'processstate', 'urbanity']), 'sa2': LayerSettings(url='https://geo.abs.gov.au/arcgis/rest/services/ASGS2021/SA2/FeatureSe

## 1.1 Selected SA4 Configuration

This section reads each team member's SA4 configuration from the config file. Later steps only process the selected SA4 areas.


In [4]:
selected_sa4_df = pd.DataFrame(
    sorted(settings.task2_import.selected_sa4_by_member.items()),
    columns=["member", "selected_sa4"],
)

config_summary = pd.DataFrame(
    [
        {"setting": "task2_import.crawl_scope", "value": settings.task2_import.crawl_scope},
        {"setting": "task3_score.score_universe", "value": settings.task3_score.score_universe},
        {"setting": "dashboard.url", "value": "https://kscii.tech"},
        {"setting": "repository.url", "value": "https://github.sydney.edu.au/xfan0282/data2001-group-assignment"},
    ]
)

display(config_summary)
display(selected_sa4_df)


,setting,value
0,task2_import.crawl_scope,selected_sa4
1,task3_score.score_universe,selected_sa4
2,dashboard.url,https://kscii.tech
3,repository.url,https://github.sydney.edu.au/xfan0282/data2001...


,member,selected_sa4
0,dabi0142,Sydney - South West Sydney
1,jzho0172,Sydney - North Sydney and Hornsby
2,xfan0282,Sydney - City and Inner South
3,xuyu8020,Sydney - Eastern Suburbs


## 2. Task 1: CSV Loading, Cleaning, and Derived Statistics

This section applies the shared cleaning workflow to the original CSV data.

> [TODO] Each team member should briefly explain the cleaning step they implemented.


In [5]:
from data2001.common.paths import resolve_project_path

raw_task1_csv = resolve_project_path(settings.outputs.raw_task1_csv)
cleaned_task1_csv = resolve_project_path(settings.outputs.processed_task1_cleaned_csv)

raw_task1_df = pd.read_csv(raw_task1_csv)
cleaned_task1_df = run_task1_cleaning(raw_task1_csv, cleaned_task1_csv)
statistics_df = run_all_task1_statistics(cleaned_task1_df)

display(raw_task1_df.head())
display(cleaned_task1_df.head())
display(statistics_df)

Statistics workflow errors:
- dabi0142_1: IndexError: single positional indexer is out-of-bounds
- dabi0142_2: IndexError: single positional indexer is out-of-bounds
- dabi0142_3: TypeError: StatisticResult.__init__() got an unexpected keyword argument 'statistic'
- dabi0142_4: TypeError: StatisticResult.__init__() got an unexpected keyword argument 'statistic'
- dabi0142_5: TypeError: StatisticResult.__init__() got an unexpected keyword argument 'statistic'


,Measure Code,Parent Description,Description,2011,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,ERP_P_20,Estimated resident population - year ended 30 ...,Estimated resident population (no.),NaN,NaN,NaN,NaN,NaN,8046748.0,8110610.0,8097062.0,8166704.0,8341199.0,8479314.0,NaN
1,ERP_21,Estimated resident population - year ended 30 ...,Population density (persons/km2),NaN,NaN,NaN,NaN,NaN,10.0,10.1,10.1,10.2,10.4,10.6,NaN
2,ERP_M_20,Estimated resident population - year ended 30 ...,Estimated resident population - males (no.),NaN,NaN,NaN,NaN,NaN,3999452.0,4030710.0,4025393.0,4059763.0,4149032.0,4217861.0,NaN
3,ERP_F_20,Estimated resident population - year ended 30 ...,Estimated resident population - females (no.),NaN,NaN,NaN,NaN,NaN,4047296.0,4079900.0,4071669.0,4106941.0,4192167.0,4261453.0,NaN
4,ERP_19,Estimated resident population - year ended 30 ...,Median age - males (years),NaN,NaN,NaN,NaN,NaN,36.8,37.2,37.7,37.7,37.5,37.5,NaN


,measure_code,parent_description,description,unit,year,value
0,ERP_P_20,Estimated resident population - year ended 30 ...,Estimated resident population,no.,2011,NaN
1,ERP_21,Estimated resident population - year ended 30 ...,Population density,persons/km2,2011,NaN
2,ERP_M_20,Estimated resident population - year ended 30 ...,Estimated resident population - males,no.,2011,NaN
3,ERP_F_20,Estimated resident population - year ended 30 ...,Estimated resident population - females,no.,2011,NaN
4,ERP_19,Estimated resident population - year ended 30 ...,Median age - males,years,2011,NaN


,member,statistic_id,title,value,unit,description
0,xfan0282,xfan0282-1,Apartment share increase among occupied privat...,2.90,percentage points,Apartment share increased 2.90pp to 21.72% (20...
1,xfan0282,xfan0282-2,"Work-from-home share growth, 2016-2021",6.42,times,Work-from-home share grew 6.42x from 4.82% to ...
2,xfan0282,xfan0282-3,"Public transport commute share drop, 2016-2021",11.98,percentage points,Public transport commute share dropped 11.98pp...
3,xfan0282,xfan0282-4,"Occupation commute distance gap, 2016",5.30,km,Commute distance gap in 2016: 5.3 km (range 14...
4,xfan0282,xfan0282-5,"Rent stress relative to mortgage stress, 2021",2.05,times,Rent stress (35.5%) is 2.05x mortgage stress (...


## 2.1 Task 1 Key Findings

This section displays the statistic values and the explanations attached to those statistics.

> [TODO] Each team member should add their explanation text in `statistics.py`.


In [ ]:
xfan0282_statistics = (
    statistics_df[statistics_df["member"] == "xfan0282"]
    .sort_values("statistic_id")
    .reset_index(drop=True)
)

pd.set_option('display.max_colwidth', 100)  # Set the maximum display width for pandas text columns to 100 characters.
display(
    xfan0282_statistics[
        ["statistic_id", "title", "value", "unit", "description"]
    ]
)


In [7]:
from pathlib import Path
import pandas as pd
import plotly.express as px

# Create output folder for report figures
fig_dir = Path("report/figures")
fig_dir.mkdir(parents=True, exist_ok=True)

# Use Task 1 statistics table
plot_df = statistics_df.copy()

# Convert value column to numeric
plot_df["value"] = pd.to_numeric(plot_df["value"], errors="coerce")
plot_df = plot_df.dropna(subset=["value"])

# Shorten long titles
plot_df["short_title"] = plot_df["title"].astype(str).str.slice(0, 35)

# Select top 10
top_stats = plot_df.sort_values("value", ascending=False).head(10)

fig = px.bar(
    top_stats,
    x="value",
    y="short_title",
    orientation="h",
    title="Top 10 Task 1 Derived Statistics",
    labels={
        "value": "Value",
        "short_title": "Statistic"
    }
)

fig.update_layout(
    yaxis=dict(autorange="reversed"),
    height=600
)

fig.show()

# Save figure for report
fig.write_image(fig_dir / "task1_top10_derived_statistics.png")

## 3. Data Source Summary

This section lists the data sources used by the downstream database import steps and their API endpoints.


In [ ]:
data_sources = pd.DataFrame(
    [
        {"source": "NSW CSV", "endpoint": "data/raw/task1/raw_data.csv"},
        {"source": "NSW POI", "endpoint": "https://maps.six.nsw.gov.au/arcgis/rest/services/public/NSW_POI/MapServer/0/query"},
        {"source": "SA2 Boundaries", "endpoint": "https://geo.abs.gov.au/arcgis/rest/services/ASGS2021/SA2/FeatureServer/0/query"},
        {"source": "SA4 Boundaries", "endpoint": "https://geo.abs.gov.au/arcgis/rest/services/ASGS2021/SA4/MapServer/0/query"},
        {"source": "SA2 Population", "endpoint": "https://geo.abs.gov.au/arcgis/rest/services/Hosted/ABS_Population_and_people_by_2021_SA2_Nov_2023/FeatureServer/1/query"},
        {"source": "SA2 Income", "endpoint": "https://geo.abs.gov.au/arcgis/rest/services/Hosted/Personal_Income_in_Australia_2022_23_SA2_2021/FeatureServer/0/query"},
    ]
)
pd.set_option('display.max_colwidth', 150)
display(data_sources)

## 4. Database Schema and Indexes

This section initializes and checks the database, then displays the current database schema and indexing design.


In [ ]:
db_setup_summary = execute_workflow_steps(engine, settings, ["init_db", "check_db"], title="Database setup")
db_setup_summary

In [ ]:
schema_summary = load_schema_summary(engine, settings)
index_summary = load_index_summary(engine, settings)

display(load_table_counts(engine, settings))
display(schema_summary.head(20))
display(index_summary)

## 5. Task 2: API Extraction and Crawl Plan

This section calls `plan-import`. The planning step uses API metadata to estimate how many SA4, SA2, and bbox requests will be processed under the current configuration.

It then runs the actual API extraction and writes the raw responses to disk. This usually takes about 60 seconds, and the API responses are cached locally as JSON files.

After that, the workflow cleans the API data, deduplicates records from bbox-based requests, and loads the results into the local database.

Finally, it shows the local JSON file summary and database state.


In [ ]:

plan_result = execute_workflow_steps(
    engine,
    settings,
    ["plan_import"],
    title="Plan import",
)

pd.DataFrame(
    [
        {
            "scope": plan_result["crawl_scope"],
            "sa4_count": plan_result["sa4_count"],
            "sa2_count": plan_result["sa2_count"],
            "sa2_bbox_requests": plan_result["sa2_bbox_requests"],
        }
    ]
)


In [ ]:
# This cell runs API extraction and database loading, so it may take a while.
task2_summary = execute_workflow_steps(
    engine,
    settings,
    ["import_boundaries", "import_poi", "import_income"],
    title="Task 2 data import",
)
task2_summary

In [ ]:
display(load_api_extraction_summary(settings))
display(load_table_counts(engine, settings))

## 6. Spatial Join Evidence

After POI records are fetched from the API using SA2 bboxes, this workflow uses PostGIS `ST_Covers(sa2.geometry, poi_clean.geometry)` to determine which SA2 polygon each POI belongs to and to remove duplicate assignments.

If a POI lies on the boundary of two SA2 polygons, the workflow assigns it to the SA2 with the lowest `sa2_code` in ascending order.

This section displays the POI assignment summary statistics.


In [ ]:
spatial_join_summary = load_spatial_join_summary(engine, settings)
display(spatial_join_summary)

## 7. Task 3: Score Calculation

The score is calculated from the POI count using a z-score and sigmoid transformation: `score_100 = sigmoid(z_poi) * 100`.

This section first displays the score input summary, then runs the score and correlation step.


In [ ]:
display(load_score_input_summary(engine, settings))

score_summary = execute_workflow_steps(engine, settings, ["compute_score"], title="Task 3 score")
score_summary

In [ ]:
scores = load_sa2_scores(
    engine,
    settings,
)
display(scores.head())
display(build_top_bottom_table(scores, n=settings.charts.top_n))

## 8. Results Analysis

The following visualisations explain the score distribution, spatial trends, Top/Bottom SA2 areas, and POI characteristics:

| Chart | Description |
|------|------|
| Score histogram | Overall distribution shape of SA2 scores, including normality, skewness, and kurtosis. |
| Top N SA2 bar chart | Ranking and exact score values for the highest-scoring areas. |
| Bottom N SA2 bar chart | Ranking and exact score values for the lowest-scoring areas. |
| POI group distribution | Count composition across POI categories such as business, education, and health. |
| Score map | Spatial distribution of SA2 scores using colour encoding. |
| SA4 summary table | Aggregated statistical summary by SA4. |
| POI point map | Spatial distribution of raw POI point locations. |

> [TODO] Add analysis and interpretation for the information shown in each chart.


In [ ]:
poi_groups = load_poi_group_counts(engine, settings)
poi_points = load_poi_points(engine, settings, limit=settings.dashboard.poi_limit)

build_score_histogram(scores, nbins=settings.charts.score_histogram_nbins).show()
build_top_sa2_bar(scores, n=settings.charts.top_n).show()
build_bottom_sa2_bar(scores, n=settings.charts.top_n).show()
build_poi_group_distribution(poi_groups).show()
build_score_choropleth_map(scores).show()

In [ ]:
sa4_summary = build_sa4_summary_table(scores, poi_points)
display(sa4_summary)

# Optional: the POI point map can be large; if the browser becomes slow, display only limited data.
build_poi_point_scatter_map(poi_points).show()

## 9. Correlation Analysis

This section uses Pearson correlation as the main test and Spearman correlation as a rank-based robustness check.


In [ ]:
score_income = load_score_income(engine, settings)
correlation_results = load_correlation_results(engine, settings)

display(load_correlation_summary(engine, settings))
display(score_income.head())
build_score_income_scatter(score_income).show()

## 10. Report Figures

This section exports the PNG figures used by the report and lists the expected output paths. If Chrome/Kaleido support is missing, run `uv run plotly_get_chrome` first.


In [ ]:
figure_summary = execute_workflow_steps(engine, settings, ["export_charts"], title="Task 4 report figures")
display(figure_summary)
display(expected_report_figure_paths(settings))

## 11. Dashboard and Repository Links

Deployed dashboard: https://kscii.tech

Repository: https://github.sydney.edu.au/xfan0282/data2001-group-assignment

Optional local dashboard command:

```bash
uv run data2001 dashboard
```


## 12. Limitations

> [TODO] List the project limitations.


## 13. Full Reproducibility Command

To rerun the full workflow from an empty database, use the following commands:

```bash
uv sync
podman compose up -d
uv run data2001 init-db
uv run data2001 run-workflow
uv run data2001 generate-figures
# optional local dashboard
uv run data2001 dashboard
```
